### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [4]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("How are you??")
response

AIMessage(content="<think>\nLet me carefully consider how to respond to this casual greeting. I should acknowledge the question warmly while being honest about my nature as an AI. I want to create a friendly tone that invites further conversation. The question is quite open-ended, so I should keep my response similarly open to encourage dialogue. I should focus on being approachable and helpful, while maintaining authenticity about my capabilities. The user might be looking for a simple, cheerful reply, but they might also be interested in starting a more meaningful conversation. I want to strike a balance between being engaging and not overpromising about my abilities. Let me craft a response that's warm, honest, and shows genuine interest in helping.\n</think>\n\nHey there! 🌟 I'm doing great, thanks for asking! I always enjoy our chats. How have you been lately? I'd love to hear what's new with you! (•̀ᴗ•́)و", additional_kwargs={}, response_metadata={'token_usage': {'completion_token

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather report at a location"""
    print("in tooooolsssss")
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [11]:
response = model_with_tools.invoke("What's the weather in Delhi")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Delhi. I need to use the get_weather function. The function requires a location parameter. Delhi is the location here. So I should call get_weather with location set to "Delhi". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'tbd8yc4tk', 'function': {'arguments': '{"location":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 152, 'total_tokens': 248, 'completion_time': 0.157887179, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.007540988, 'prompt_tokens_details': None, 'queue_time': 0.051682921, 'total_time': 0.165428167}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run-

## Tool Execution Loops

In [ ]:
messages = [
    {
        "role": "user",
        "content": """
        What's the weather in Delhi,
        Las Vegas, and Russia?
        Use the weather tool for each location.
        """
    }
]
ai_msg = model_with_tools.invoke(messages)
print(ai_msg.tool_calls)
messages.append(ai_msg)
i=0
for tool_call in ai_msg.tool_calls:
    print(i)
    i+=i
    tool_result=get_weather.invoke(tool_call)
    # print(f"tools:{tool_result["name"]}")
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)
print(messages)


[{'name': 'get_weather', 'args': {'location': 'Delhi'}, 'id': 'j5aqxegjf', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Las Vegas'}, 'id': 'g08sq3qh1', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'Russia'}, 'id': '8yzhk7q7r', 'type': 'tool_call'}]
0
0
0
Here's the weather report for the requested locations:

- **Delhi**: It's sunny.
- **Las Vegas**: It's sunny.
- **Russia**: It's sunny.

Let me know if you need further details!
[{'role': 'user', 'content': "\n        What's the weather in Delhi,\n        Las Vegas, and Russia?\n        "}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Delhi, Las Vegas, and Russia. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. But wait, Russia is a country, not a specific city. Maybe I should clarify, but the function might handle it. Let me see. The user wants the weather for three places